# 🚀 Обучение крипто-бота на бесплатном GPU (Kaggle)

Этот ноутбук настроен для обучения модели на **5 топовых криптовалютах** и **7 таймфреймах**.

**Важно:** Перед запуском убедитесь, что включен GPU:
1. Нажмите `Settings` (справа) → `Accelerator` → Выберите `GPU T4 x2`
2. Убедитесь, что включен `Internet` (нужен для загрузки данных с Yahoo Finance)
3. Сохраните ноутбук (`Save Version`) для запуска в фоне

## 1. Установка зависимостей

In [ ]:
!pip install yfinance --quiet
print("✅ Зависимости установлены")

## 2. Проверка окружения

In [ ]:
import os
import torch
import pandas as pd
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Устройство: {device}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")

os.makedirs('/kaggle/working/models', exist_ok=True)
os.makedirs('/kaggle/working/data', exist_ok=True)
print("✅ Папки созданы")

## 3. Конфигурация

In [ ]:
CONFIG = {
    'symbols': ['BTC-USD', 'ETH-USD', 'BNB-USD', 'SOL-USD', 'XRP-USD'],
    'timeframes': ['5m', '15m', '1h', '4h', '12h', '1d', '1wk'],
    'model': {
        'hidden_size': 128,
        'num_layers': 2,
        'dropout': 0.2,
    },
    'training': {
        'epochs': 50,
        'batch_size': 32,
        'learning_rate': 0.001,
        'test_split': 0.2,
    },
    'lookback_periods': {
        '5m': 200, '15m': 150, '1h': 100,
        '4h': 80, '12h': 60, '1d': 50, '1wk': 30,
    }
}

print(f"📋 Криптовалюты: {CONFIG['symbols']}")
print(f"⏱️ Таймфреймы: {CONFIG['timeframes']}")

## 4. Загрузка данных (Yahoo Finance)

In [ ]:
import yfinance as yf

def fetch_crypto_data(symbol: str, timeframe: str, limit: int = 1000):
    try:
        ticker = yf.Ticker(symbol)
        df = ticker.history(period='max', interval=timeframe)
        
        if df.empty:
            return None
            
        df = df.reset_index()
        df.columns = [c.lower() for c in df.columns]
        if 'date' in df.columns:
            df.rename(columns={'date': 'timestamp'}, inplace=True)
            
        df = df.tail(limit).reset_index(drop=True)
        return df
    except Exception as e:
        print(f"⚠️ Ошибка {symbol} ({timeframe}): {e}")
        return None

all_data = {}
for symbol in CONFIG['symbols']:
    print(f"\n📥 Загрузка {symbol}...")
    symbol_data = {}
    for tf in CONFIG['timeframes']:
        limit = CONFIG['lookback_periods'].get(tf, 1000) * 2
        df = fetch_crypto_data(symbol, tf, limit=limit)
        if df is not None and not df.empty:
            symbol_data[tf] = df
            print(f"   ✅ {tf}: {len(df)} свечей")
        else:
            print(f"   ❌ {tf}: нет данных")
    if symbol_data:
        all_data[symbol] = symbol_data

print(f"\n✅ Загружено данных для {len(all_data)} криптовалют")

## 5. Инжиниринг признаков

In [ ]:
def add_features(df):
    df = df.copy()
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))
    df['sma_20'] = df['close'].rolling(window=20).mean()
    df['sma_50'] = df['close'].rolling(window=50).mean()
    df['ema_12'] = df['close'].ewm(span=12, adjust=False).mean()
    df['ema_26'] = df['close'].ewm(span=26, adjust=False).mean()
    df['macd'] = df['ema_12'] - df['ema_26']
    df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    bb_std = df['close'].rolling(window=20).std()
    df['bb_width'] = (bb_std * 2) / df['close']
    df['volatility'] = df['close'].pct_change().rolling(window=14).std()
    df['returns'] = df['close'].pct_change()
    df['volume_sma'] = df['volume'].rolling(window=20).mean()
    df['volume_ratio'] = df['volume'] / df['volume_sma']
    df['target'] = (df['close'].shift(-1) > df['close']).astype(int)
    df.dropna(inplace=True)
    return df

all_data_featured = {}
feature_columns = None

for symbol, tf_data in all_data.items():
    print(f"\n🔧 Обработка {symbol}...")
    featured_symbol_data = {}
    for tf, df in tf_data.items():
        featured_df = add_features(df)
        featured_symbol_data[tf] = featured_df
        print(f"   ✅ {tf}: {len(featured_df)} строк")
        if feature_columns is None:
            feature_columns = [c for c in featured_df.columns if c not in ['timestamp', 'target']]
    all_data_featured[symbol] = featured_symbol_data

print(f"\n✅ Признаков: {len(feature_columns)}")
print(f"Список: {feature_columns}")

## 6. Модель Multi-Timeframe LSTM

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class AttentionLayer(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)
    
    def forward(self, lstm_outputs):
        attention_weights = F.softmax(self.attention(lstm_outputs), dim=1)
        context = torch.sum(attention_weights * lstm_outputs, dim=1)
        return context, attention_weights

class MultiTimeframeLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout, num_timeframes):
        super().__init__()
        self.num_timeframes = num_timeframes
        self.lstms = nn.ModuleList([
            nn.LSTM(input_size, hidden_size, num_layers, 
                   batch_first=True, dropout=dropout if num_layers > 1 else 0)
            for _ in range(num_timeframes)
        ])
        self.attention = AttentionLayer(hidden_size)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        batch_size = x.size(0)
        lstm_outputs = []
        for i in range(self.num_timeframes):
            lstm_out, _ = self.lstms[i](x[:, i, :, :])
            lstm_outputs.append(lstm_out[:, -1, :])
        lstm_stack = torch.stack(lstm_outputs, dim=1)
        context, _ = self.attention(lstm_stack)
        output = self.fc(context)
        return output

num_tf = len(CONFIG['timeframes'])
input_size = len(feature_columns)
model = MultiTimeframeLSTM(
    input_size=input_size,
    hidden_size=CONFIG['model']['hidden_size'],
    num_layers=CONFIG['model']['num_layers'],
    dropout=CONFIG['model']['dropout'],
    num_timeframes=num_tf
).to(device)

print(f"✅ Модель создана: {num_tf} таймфреймов, {input_size} признаков")

## 7. Подготовка данных

In [ ]:
from sklearn.preprocessing import MinMaxScaler

def prepare_data(symbol_data_dict, lookback_periods, features):
    scalers = {}
    timeframe_sequences = {}
    
    for tf, df in symbol_data_dict.items():
        scaler = MinMaxScaler()
        scaled_data = scaler.fit_transform(df[features])
        scalers[tf] = scaler
        lookback = lookback_periods.get(tf, 60)
        sequences = []
        targets = []
        for i in range(lookback, len(scaled_data) - 1):
            sequences.append(scaled_data[i-lookback:i])
            targets.append(df['target'].iloc[i])
        timeframe_sequences[tf] = {
            'sequences': np.array(sequences),
            'targets': np.array(targets),
        }
    
    min_len = min(len(seq['sequences']) for seq in timeframe_sequences.values())
    X_list = []
    y_list = []
    timeframes = list(timeframe_sequences.keys())
    for i in range(min_len):
        sample = [timeframe_sequences[tf]['sequences'][i] for tf in timeframes]
        X_list.append(sample)
        y_list.append(timeframe_sequences[timeframes[0]]['targets'][i])
    
    return np.array(X_list), np.array(y_list), scalers

print("✅ Функция подготовки данных готова")

## 8. Обучение

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

results = {}
features = feature_columns

for symbol, tf_data in all_data_featured.items():
    print(f"\n{'='*60}")
    print(f"🚀 Обучение: {symbol}")
    print('='*60)
    
    X, y, scalers = prepare_data(tf_data, CONFIG['lookback_periods'], features)
    print(f"📦 Данные: {X.shape}, Целевая: {y.shape}")
    
    split_idx = int(len(X) * (1 - CONFIG['training']['test_split']))
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    
    train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['training']['batch_size'], shuffle=True)
    
    model = MultiTimeframeLSTM(
        input_size=len(features),
        hidden_size=CONFIG['model']['hidden_size'],
        num_layers=CONFIG['model']['num_layers'],
        dropout=CONFIG['model']['dropout'],
        num_timeframes=len(CONFIG['timeframes'])
    ).to(device)
    
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['training']['learning_rate'])
    
    best_loss = float('inf')
    for epoch in range(CONFIG['training']['epochs']):
        model.train()
        total_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            output = model(batch_X).squeeze()
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), f'/kaggle/working/models/{symbol.replace("-", "_")}_best.pth')
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{CONFIG['training']['epochs']}, Loss: {avg_loss:.4f}")
    
    model.eval()
    with torch.no_grad():
        X_test_t = torch.FloatTensor(X_test).to(device)
        preds = (model(X_test_t).squeeze() > 0.5).cpu().numpy()
        accuracy = (preds == y_test).mean()
    
    results[symbol] = {'best_loss': best_loss, 'accuracy': accuracy}
    print(f"✅ Точность: {accuracy:.2%}")

print(f"\n{'='*60}")
print("🎉 Обучение завершено!")
print('='*60)
for sym, res in results.items():
    print(f"{sym}: Loss={res['best_loss']:.4f}, Acc={res['accuracy']:.2%}")

## 9. Сохранение результатов

In [ ]:
import json

with open('/kaggle/working/training_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

print("✅ Метрики сохранены в training_metrics.json")
print("\n📁 Модели сохранены в /kaggle/working/models/")
print("💡 Нажмите 'Save Version' для сохранения результатов")